In [124]:
import pandas as pd
import numpy as np

from control_totals.util import Pipeline

p = Pipeline('configs')

In [247]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
tables

['/adjusted_emp_change_targets_calculations',
 '/adjusted_emp_change_targets_no_res_con',
 '/adjusted_emp_change_targets_no_res_con_calculations',
 '/adjusted_emp_change_targets_res_con',
 '/adjusted_emp_change_targets_res_con_calculations',
 '/adjusted_emp_targets_no_res_con',
 '/adjusted_king_targets',
 '/adjusted_total_pop_change_targets',
 '/adjusted_units_change_targets',
 '/block_2010_control_area_xwalk',
 '/block_control_area_xwalk',
 '/blocks',
 '/blocks_2010',
 '/control_areas',
 '/control_target_xwalk',
 '/control_totals',
 '/dec_block_data',
 '/decennial_by_control_area',
 '/employment_2018_by_control_area',
 '/employment_2019_by_control_area',
 '/employment_2020_by_control_area',
 '/extrapolated_targets',
 '/king_targets',
 '/kitsap_targets',
 '/ofm_block_2019',
 '/ofm_block_2019_by_control_area',
 '/ofm_parcel_control_area_xwalk',
 '/ofm_parcelized_2018',
 '/ofm_parcelized_2018_by_control_area',
 '/ofm_parcelized_2019',
 '/ofm_parcelized_2019_by_control_area',
 '/ofm_parce

In [ ]:
target_year = p.settings['targets_end_year']

# load targets and join to rgids
df = p.get_table('adjusted_emp_change_targets_no_res_con')
xwalk = p.get_table('target_rgid_xwalk')[['target_id','rgid']]
df = df.merge(xwalk, on='target_id', how='left')
# split snohomish to seperate df
snohomish_df = df[df['county_id'] == 53061].copy()
remainder_df = df[df['county_id'] != 53061].copy()
# get hard coded Snohomish County employment totals by rgid from settings.yaml
snohomish_emp_totals = pd.Series(p.settings[f'snohomish_emp_target_totals'])
# calculate preliminary employment by rgid for Snohomish County
prelim_emp_by_rgid = snohomish_df.groupby('rgid')[f'emp_{target_year}'].sum()
# calculate adjustment ratio
adj_ratio = snohomish_emp_totals / prelim_emp_by_rgid
# apply adjustment ratio to snohomish_df
snohomish_df[f'emp_{target_year}'] = (
    round(snohomish_df['rgid'].map(adj_ratio) * snohomish_df[f'emp_{target_year}'])
)


In [128]:
snohomish_df.query('target_id == 125')

,target_id,start,emp_chg,emp_chg_adj,county_id,control_id_2020,control_name_2020,Emp_TotNoMil_2020,Emp_ConRes_2020,TotEmpNoMil-ResCon_2020,Emp_Military_2020,Emp_Total_2020,emp_2044,rgid
93,125,2019,67340.0,66710.80687,53061,529,EverettNaval Station Everett,96227,3710,92517,2211,98438,164981.0,1


In [ ]:
# combine snohomish_df and remainder_df back into a single dataframe
df = pd.concat([snohomish_df, remainder_df], ignore_index=True)

In [126]:
df.query('target_id == 125')

,target_id,start,emp_chg,emp_chg_adj,county_id,control_id_2020,control_name_2020,Emp_TotNoMil_2020,Emp_ConRes_2020,TotEmpNoMil-ResCon_2020,Emp_Military_2020,Emp_Total_2020,res_con_pct,res_con_target_pct,emp_chg_cnty_sum,res_con_emp_chg_cnty_sum,res_con_emp_chg_target,emp_chg_adj_res_con,emp_2044
56,125,2019,67340.0,66710.80687,53061,529,EverettNaval Station Everett,96227,3710,92517,2211,98438,0.038555,2596.271317,171820.0,4810.96,874.600924,67585.407794,163812


rgid
1    162937.806870
2     76298.273506
3    156801.152628
4     44528.192278
5     22717.307064
6     30946.754601
Name: emp_2044, dtype: float64